In [1]:
import project_path
from rsatools import *
from math import gcd, lcm
from rsatools import modinverse
from sympy import mod_inverse

In [2]:
rng = prime_rng(8675_309)

p = rng.randprime(2**31, 2**32)
q = rng.randprime(2**31, 2**32)

n = p*q

lamb = carmichael(p, q)

e = 257
d = inverse257(lamb)

secrets = [p, q, n, lamb, e, d]

for num in secrets:
    bitlength = len(str(bin(num)))-2
    print(f"{bitlength:2.0f}    {num}")

32    3721615301
32    2961235187
64    11020578181798796287
59    423868391350613300
 9    257
58    222654602460438905


In [3]:
print(p-1)
print(q-1)
print((q-1)//26)

3721615300
2961235186
113893661


In [4]:
gcd(p-1, q-1)

26

In [5]:
((p-1)//26)*(q-1)

423868391350613300

okay wait I checked a different online calculator and it matched python's answer. I'm going to pull out my graphing calculator and check that way.  I trust texas instruments more than anyone.

actually the number is so big it doesnt display on the scientific calculator.  :(

Okay the online calculator was wrong.  So the lcm function is fine.  But we are still concerned that e and d are not actually inverses?

In [6]:
e * d

57222232832332798585

In [7]:
57222232832332798585 % 423868391350613300

3085

so something is going wrong with the `inverse257()` function, maybe?  let's see if it can find the proper inverse using euler's totient rather than Carmichaels

In [8]:
dprime = inverse257((p-1)*(q-1))
print(dprime)
print(d)

8276154038122091777
222654602460438905


d prime is larger, as we might expect

In [9]:
(dprime * 257) % n

11020576892008699585

Oh no, something has gone horribly wrong

In [10]:
modinverse(257, lamb)

222654602460438905

so something is wrong with our implementation of modinverse, and therefore inverse257

In [11]:
mod_inverse(257, lamb)

222654602460438893

In [12]:
(mod_inverse(257, lamb) * 257) % lamb

1

okay so sympy's version works.  my version is off by just a tiny bit.  why does that happen?

In [31]:
def modinverse_test(a: int, n: int) -> int:
    """Return the multiplicative inverse of `a` modulo `n`.
    
    This function uses the extended Euclidean algorithm to solve 
    for `gcd(a,n)` and minimal Bezout Coefficients (BC). Assuming 
    that `gcd(a,n)=1`, the positive minimal BC of `a` is exactly the
    multiplicative inverse of `a` mod `n`. After the `while()` loop, 
    `r0 = gcd(a,n)`, `s0` is a minimal BC for `a`, and `t0` is the 
    relevant BC for `n`.  The sign of the inputs is not used, so `a` 
    and `-a` give the same output.  `modinverse(a, b)` will always 
    output a positive number.  Using 0 as either input will probably
    break something.
    """

    if abs(a)==abs(n):
        return 1        # this is the trivial case of a=b
                        #
    elif abs(a)>abs(n): # WANT: sa + tb = gcd(a,b) AND r0=a
        r0 = abs(a)
        r1 = abs(n)
        s0 = 1
        t0 = 0
        s1 = 0
        t1 = 1
                        #
    elif abs(a)<abs(n): # WANT: sa + tb = gcd(a,b) AND r0=b
        r0 = abs(n)     # to achieve this, we need to flip
        r1 = abs(a)     # the definitions of s and t
        s0 = 0
        t0 = 1
        s1 = 1
        t1 = 0

    else:
        raise RuntimeError("This is impossible.")
                        #
    while not r1==0:    # Apply the extended Euclidean alg.
        q1 = r0 // r1
        r2 = r0 % r1
        s2 = s0 - (q1*s1)
        t2 = t0 - (q1*t1)
        print(q1, r0, r1, r2, s1, t1, "", sep="\n")
        r0 = r1
        r1 = r2
        s0 = s1
        s1 = s2
        t0 = t1
        t1 = t2
                        #
    if s0>0:            # now ensure positive output
        return s0  
        # s is already positive

    elif s0<0:
        print(s0, r0, sep = "\n")
        return s0+int(abs(n)/r0)
        # we find the *other* minimal bezout coefficients
        # so that s is positive and t is negative

    else:
        raise RuntimeError("This should never happen.")

In [32]:
modinverse_test(257, lamb)

1649293351558806
423868391350613300
257
158
1
0

1
257
158
99
-1649293351558806
1

1
158
99
59
1649293351558807
-1

1
99
59
40
-3298586703117613
2

1
59
40
19
4947880054676420
-3

2
40
19
2
-8246466757794033
5

9
19
2
1
21440813570264486
-13

2
2
1
0
-201213788890174407
122

-201213788890174407
1


222654602460438905

In [27]:
print(201213788890174407*257)
print(122*lamb)
print(201213788890174407*257-122*lamb)

51711943744774822599
51711943744774822600
-1


so it correctly gets to the minimal bezout coefficients, but must fail at converting them

Yeah it looks like I fucked it up.  So we fix it then:

In [41]:
201213788890174407/e

782933030701067.8

In [42]:
-201213788890174407+(782933030701067*e)

-188

In [43]:
122-782933030701067*lamb

-331860564258521604846299414390978

goddamnit

In [44]:
for k in range(-1, 10):
    x = -201213788890174407 + (k * lamb)
    y = 122 - (k* 257)
    print(x, y)

-625082180240787707 379
-201213788890174407 122
222654602460438893 -135
646522993811052193 -392
1070391385161665493 -649
1494259776512278793 -906
1918128167862892093 -1163
2341996559213505393 -1420
2765864950564118693 -1677
3189733341914731993 -1934
3613601733265345293 -2191


In [45]:
(222654602460438893 * 257) - (135 * lamb)

1

tf?

In [46]:
mod_inverse(257, lamb)

222654602460438893

In [47]:
modinverse(257, lamb)

222654602460438905

tf??

In [56]:
def modinverse3(a: int, n: int) -> int:
    """Return the multiplicative inverse of `a` modulo `n`.
    
    This function uses the extended Euclidean algorithm to solve 
    for `gcd(a,n)` and minimal Bezout Coefficients (BC). Assuming 
    that `gcd(a,n)=1`, the positive minimal BC of `a` is exactly the
    multiplicative inverse of `a` mod `n`. After the `while()` loop, 
    `r0 = gcd(a,n)`, `s0` is a minimal BC for `a`, and `t0` is the 
    relevant BC for `n`.  The sign of the inputs is not used, so `a` 
    and `-a` give the same output.  `modinverse(a, b)` will always 
    output a positive number.  Using 0 as either input will probably
    break something.
    """

    if abs(a)==abs(n):
        return 1        # this is the trivial case of a=b
                        #
    elif abs(a)>abs(n): # WANT: sa + tb = gcd(a,b) AND r0=a
        r0 = abs(a)
        r1 = abs(n)
        s0 = 1
        t0 = 0
        s1 = 0
        t1 = 1
                        #
    elif abs(a)<abs(n): # WANT: sa + tb = gcd(a,b) AND r0=b
        r0 = abs(n)     # to achieve this, we need to flip
        r1 = abs(a)     # the definitions of s and t
        s0 = 0
        t0 = 1
        s1 = 1
        t1 = 0

    else:
        raise RuntimeError("This is impossible.")
                        #
    while not r1==0:    # Apply the extended Euclidean alg.
        q1 = r0 // r1
        r2 = r0 % r1
        s2 = s0 - (q1*s1)
        t2 = t0 - (q1*t1)
        r0 = r1
        r1 = r2
        s0 = s1
        s1 = s2
        t0 = t1
        t1 = t2
                        #
    if s0>0:            # now ensure positive output
        return s0  
        # s is already positive

    elif s0<0:
        print(s0, r0)
        print(s0+n)
        print(n/r0)
        return s0+int(n/r0)
        # we find the *other* minimal bezout coefficients
        # so that s is positive and t is negative

    else:
        raise RuntimeError("This should never happen.")


In [57]:
modinverse3(257, lamb)

-201213788890174407 1
222654602460438893
4.238683913506133e+17


222654602460438905

# ITS FUCKING FLOATING POINT ERROR.  I'M GOING TO SHOOT MYSELF

at least the solution is easy.  we just change / to // and everything will be okay